In [1]:
import google.generativeai as genai
import json
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 1\DAY 1\.env")

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-3.6-flash")

C:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\PC\AppData\Local\Temp\ipykernel_1236\1328381268.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
%pip install -U google-genai

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ------------------- -------------------- 0.5/1.1 MB 1.2 MB/s eta 0:00:01
   ---------------------------- ----------- 0.8/1.1 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.6 MB/s  0:00:01
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.17.0
    Uninstalling google-genai-2.17.0:
      Successfully uninstalled google-genai-2.17.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
paragraph = """
Marie Curie, a physicist and chemist, conducted pioneering research 
on radioactivity at the University of Paris. Her work later influenced 
organizations like CERN.
"""

prompt = f"""
Extract all entities from the following text.
Classify each entity as one of: PERSON, ORG, CONCEPT.
Return ONLY valid JSON in this format, with no extra text:
[{{"entity": "...", "type": "..."}}]

Text: {paragraph}
"""

In [4]:
response = model.generate_content(prompt)
print(response.text)

[
  {"entity": "Marie Curie", "type": "PERSON"},
  {"entity": "physicist", "type": "CONCEPT"},
  {"entity": "chemist", "type": "CONCEPT"},
  {"entity": "radioactivity", "type": "CONCEPT"},
  {"entity": "University of Paris", "type": "ORG"},
  {"entity": "CERN", "type": "ORG"}
]


In [5]:
raw_text = response.text.strip()

# Remove markdown code fences if present
if raw_text.startswith("```"):
    raw_text = raw_text.split("```")[1]
    raw_text = raw_text.removeprefix("json").strip()

entities = json.loads(raw_text)

for e in entities:
    print(e["entity"], "->", e["type"])

Marie Curie -> PERSON
physicist -> CONCEPT
chemist -> CONCEPT
radioactivity -> CONCEPT
University of Paris -> ORG
CERN -> ORG


In [6]:
def extract_entities(text):
    prompt = f"""
Extract all entities from the following text.
Classify each entity as one of: PERSON, ORG, CONCEPT.
Return ONLY valid JSON in this format, with no extra text:
[{{"entity": "...", "type": "..."}}]

Text: {text}
"""
    response = model.generate_content(prompt)
    raw_text = response.text.strip()

    if raw_text.startswith("```"):
        raw_text = raw_text.split("```")[1]
        raw_text = raw_text.removeprefix("json").strip()

    return json.loads(raw_text)


# Test it with a new paragraph
test_paragraph = "Google was founded by Larry Page and Sergey Brin at Stanford University."
result = extract_entities(test_paragraph)

for e in result:
    print(e["entity"], "->", e["type"])

Google -> ORG
Larry Page -> PERSON
Sergey Brin -> PERSON
Stanford University -> ORG


conclusion = """
## Conclusion — Day 23: Entity Extraction with Gemini

Today I learned how to extract entities (people, organizations, concepts) 
from unstructured text using Gemini.

### Key steps:
1. Loaded the Gemini API using my `.env` file.
2. Wrote a prompt that gave Gemini a fixed schema (PERSON, ORG, CONCEPT) 
   and asked for JSON-only output.
3. Called `model.generate_content()` and inspected the raw response.
4. Learned that Gemini often wraps JSON in ```json fences even when told not to, 
   and cleaned the response before parsing.
5. Parsed the cleaned text into a Python list using `json.loads()`.
6. Wrapped the whole flow into a reusable `extract_entities(text)` function 
   and tested it on a new paragraph.

### Why this matters:
Extracted entities become the foundation for building a knowledge graph — 
each entity can become a **node** (connecting back to Day 22's graph theory), 
and future relation extraction can define the **edges** between them. 
This is a direct building block toward my Graph RAG goal.
"""

with open("conclusion.md", "w") as f:
    f.write(conclusion)

print("Conclusion saved.")